# Natural Language Processing with Disaster Tweets

## Project plan

In this notebook, we will build a binary text classifier that predicts whether a tweet is about a real disaster.

### Workflow
1. Data loading
2. Quick inspection
3. Text preprocessing
4. Feature extraction with TF-IDF
5. Model training and evaluation
6. Final conclusions

### Goal
We want a solution that is both:
- simple enough to understand
- strong enough to serve as a baseline

The main label is:
- `1` = disaster tweet
- `0` = non-disaster tweet

## 1. Data loading

We use the Kaggle competition dataset **Natural Language Processing with Disaster Tweets**.

The training file is loaded locally from `data/train.csv`.

In [48]:
# Import the libraries used in the first stage of the project.
from pathlib import Path
import pandas as pd

# Point to the local data folder that already contains train.csv.
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"

print(f"Project folder: {Path.cwd()}")
print(f"Train file exists: {TRAIN_PATH.exists()}")

Project folder: /Users/egortatsiy/Desktop/Masterschool/Disaster Tweets Project
Train file exists: True


### Load the training data

We load `train.csv` into a pandas DataFrame.

The `train.csv` file contains both the tweet text and the target label, so it is the main file we need for model development.


In [49]:
# Load the training set from the local data folder.
train_df = pd.read_csv(TRAIN_PATH)

# Show the first rows to confirm that the file loaded correctly.
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


### Quick inspection

Before modeling, we check the shape, columns, and basic data types.

This helps us confirm that the dataset was loaded correctly and that the target column is available.

In [50]:
print(f"Dataset shape: {train_df.shape}")
print("\nColumns:")
print(train_df.columns.tolist())

print("\nMissing values per column:")
print(train_df.isna().sum())

print("\nData types:")
print(train_df.dtypes)

train_df.info()

Dataset shape: (7613, 5)

Columns:
['id', 'keyword', 'location', 'text', 'target']

Missing values per column:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Data types:
id           int64
keyword     object
location    object
text        object
target       int64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


## 2. Data quality check

Before modeling, we check whether the same tweet appears with different labels.

If the training set contains conflicting duplicates, that is an important source of noise and it should be reported in the notebook.

This check is implemented in the next code cell, where we group by `text` and count how many unique `target` values each tweet has.


In [51]:
# Check for duplicate tweet texts with different target values.
duplicate_summary = (
    train_df.groupby('text')['target']
    .nunique()
    .reset_index(name='target_variants')
)

conflicting_texts = duplicate_summary[duplicate_summary['target_variants'] > 1]

print(f'Number of unique tweet texts: {train_df["text"].nunique()}')
print(f'Tweets with conflicting labels: {len(conflicting_texts)}')

if len(conflicting_texts) > 0:
    display(conflicting_texts.head(10))


Number of unique tweet texts: 7503
Tweets with conflicting labels: 18


,text,target_variants
24,#Allah describes piling up #wealth thinking it...,2
284,#foodscare #offers2go #NestleIndia slips into ...,2
610,.POTUS #StrategicPatience is a strategy for #G...,2
2679,CLEARED:incident with injury:I-495 inner loop...,2
2748,Caution: breathing may be hazardous to your he...,2
3589,He came to a land which was engulfed in tribal...,2
3616,Hellfire is surrounded by desires so be carefu...,2
3618,Hellfire! We donÛªt even want to think about ...,2
3750,I Pledge Allegiance To The P.O.P.E. And The Bu...,2
4193,In #islam saving a person is equal in reward t...,2


## 2.1. How to handle conflicting labels

The safest default is to remove tweet texts that appear with both labels from the training data.

Why this is a good default:
- it avoids teaching the model contradictory examples
- it keeps the dataset clean for fair evaluation
- it is easy to explain in the presentation

Important detail: the number of removed rows can be larger than the number of conflicting tweet texts.
For example, if 18 unique tweet texts are conflicting but some of them appear multiple times in the dataset,
then removing every occurrence of those texts can delete 55 rows in total.

We can still mention that a more advanced approach would be manual review or majority voting if there were many duplicates.


In [52]:
# Remove all rows whose text has conflicting labels.
# This keeps the baseline cleaner and avoids contradiction during training.
if len(conflicting_texts) > 0:
    conflict_text_values = set(conflicting_texts['text'])
    clean_train_df = train_df[~train_df['text'].isin(conflict_text_values)].copy()
else:
    clean_train_df = train_df.copy()

print(f'Original rows: {len(train_df)}')
print(f'Rows after removing conflicts: {len(clean_train_df)}')
print(f'Removed rows: {len(train_df) - len(clean_train_df)}')


Original rows: 7613
Rows after removing conflicts: 7558
Removed rows: 55


## 3. Preprocessing plan

For the first baseline, we will keep preprocessing simple and explainable.

The preprocessing pipeline will be written as separate steps so the logic is easy to follow:

1. convert text to lowercase
2. remove URLs
3. remove punctuation, digits, and other non-alphabetic characters
4. tokenize the text into words
5. remove stop words
6. apply a lightweight lemmatization step
7. join the cleaned tokens back into a single text string

After that, TF-IDF will convert the cleaned text into numeric features for the classical models.


In [53]:
import json
import re

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Explicit preprocessing steps for the presentation-friendly baseline.
def normalize_lower(text):
    return str(text).lower()

def remove_urls(text):
    return re.sub(r'http\S+|www\.\S+', ' ', text)

def remove_punctuation_and_digits(text):
    return re.sub(r'[^a-z\s]', ' ', text)

def tokenize_text(text):
    return text.split()

def remove_stopwords(tokens):
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

def simple_lemmatize_token(token):
    # Lightweight rule-based normalization because nltk/spacy are not available here.
    if len(token) <= 3:
        return token
    if token.endswith('ies') and len(token) > 4:
        return token[:-3] + 'y'
    if token.endswith('ing') and len(token) > 5:
        return token[:-3]
    if token.endswith('ed') and len(token) > 4:
        return token[:-2]
    if token.endswith('s') and not token.endswith('ss') and len(token) > 3:
        return token[:-1]
    return token

def lemmatize_tokens(tokens):
    return [simple_lemmatize_token(token) for token in tokens]

def preprocess_text(text):
    text = normalize_lower(text)
    text = remove_urls(text)
    text = remove_punctuation_and_digits(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    return ' '.join(tokens)

# Preserve the cleaned dataset for the classical baselines.
baseline_train_df = clean_train_df.copy()
baseline_train_df['preprocessed_text'] = baseline_train_df['text'].apply(preprocess_text)
baseline_train_df[['text', 'preprocessed_text', 'target']].head()


,text,clean_text,target
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,1
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,1
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,1
3,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...,1
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,1


## 4. Train-test split and baseline models

We use a stratified split so both classes stay represented in train and validation sets.

Then we compare a few simple models using the same TF-IDF representation and the same `F1` metric.


In [54]:
X = baseline_train_df['preprocessed_text']
y = baseline_train_df['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

models = {
    'logreg': {
        'model': LogisticRegression(max_iter=2000, C=1.0, solver='liblinear', class_weight='balanced'),
        'vectorizer': TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95),
    },
    'svm': {
        'model': LinearSVC(C=1.0),
        'vectorizer': TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95),
    },
    'random_forest': {
        'model': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample'),
        'vectorizer': TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.95, max_features=5000),
    },
}

results = []

for name, config in models.items():
    pipeline = Pipeline([
        ('tfidf', config['vectorizer']),
        ('model', config['model']),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)
    f1 = f1_score(y_valid, preds)
    results.append({
        'model': name,
        'f1': f1,
        'vectorizer': json.dumps(config['vectorizer'].get_params(), default=str),
        'model_params': json.dumps(config['model'].get_params(), default=str),
    })
    print(f'\nModel: {name}')
    print(f'F1: {f1:.4f}')
    print(classification_report(y_valid, preds))
    print('Confusion matrix:')
    print(confusion_matrix(y_valid, preds))

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
results_df



Model: logreg
F1: 0.7873
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       869
           1       0.80      0.78      0.79       654

    accuracy                           0.82      1523
   macro avg       0.82      0.81      0.82      1523
weighted avg       0.82      0.82      0.82      1523

Confusion matrix:
[[742 127]
 [147 507]]

Model: svm
F1: 0.7695
              precision    recall  f1-score   support

           0       0.82      0.85      0.83       869
           1       0.79      0.75      0.77       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523

Confusion matrix:
[[741 128]
 [165 489]]

Model: random_forest
F1: 0.7433
              precision    recall  f1-score   support

           0       0.79      0.86      0.82       869
           1       0.79      0.70      0.74       654

    accuracy      

,model,f1,vectorizer,model_params
0,logreg,0.787267,"{""analyzer"": ""word"", ""binary"": false, ""decode_...","{""C"": 1.0, ""class_weight"": ""balanced"", ""dual"":..."
1,svm,0.769473,"{""analyzer"": ""word"", ""binary"": false, ""decode_...","{""C"": 1.0, ""class_weight"": null, ""dual"": ""auto..."
2,random_forest,0.743320,"{""analyzer"": ""word"", ""binary"": false, ""decode_...","{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w..."


## 5. Next steps

Planned extensions:

- keep the explicit preprocessing pipeline for the classical models
- compare the baseline TF-IDF models using the cleaned text
- test a pretrained RoBERTa-based model in a fast single split setup first
- only move to heavier cross-validation if the quick run looks promising

Important note: because the training data contains conflicting labels for identical tweet texts, a perfect `F1 = 1.0` on a fair train/validation setup is not a realistic expectation. The right goal is to push performance as high as possible while documenting the noise in the labels.


## 6. Fast RoBERTa experiment (archived)

The first quick RoBERTa run is kept here as an archived baseline result.

Archived result: `F1 = 0.7932` with best threshold `0.45`.


## 6.0. RoBERTa v1 code (archived)

The original v1 transformer code has been archived to keep the notebook focused on the active DeBERTa experiment below.


## 7. Final summary

This notebook now contains two layers of results:

1. classical baselines with explicit preprocessing, logged `F1` scores, and parameters
2. the active DeBERTa v3 base sanity run with softer text cleaning and keyword context

Current results so far:

- Logistic Regression baseline: `F1 = 0.7873`
- Linear SVM baseline: `F1 = 0.7695`
- Random Forest baseline: `F1 = 0.7433`
- Fast RoBERTa validation run: `F1 = 0.7932` with best threshold `0.45`
- RoBERTa v2 validation run: `F1 = 0.8116` with best threshold `0.35`
- RoBERTa v3 validation run: `F1 = 0.8049` with best threshold `0.420`
- The next active run is DeBERTa v3 base

For the presentation, emphasize the following:
- the dataset contains conflicting duplicate texts
- cleaning those conflicts is part of the modeling decision
- the preprocessing pipeline is now explicit and easy to explain
- the active DeBERTa run is intended as a higher-capacity transformer sanity check
- the final result should be presented as a validated F1, not just a single split score


## 6.1. DeBERTa v3 base sanity run

This is the active transformer experiment. We keep the preprocessing soft, include the keyword as extra context, and test `microsoft/deberta-v3-base` on a single train/validation split first.

If this improves on RoBERTa v2, then the next step is 5-fold cross-validation.

Suggested setup:
- `microsoft/deberta-v3-base`
- `max_len = 192`
- `batch_size = 4`
- `epochs = 2`
- `learning_rate = 1e-5`
- `weight_decay = 0.01`
- `warmup_ratio = 0.1`
- threshold tuning on validation probabilities


In [ ]:
# DeBERTa v3 base: soft cleaning, keyword context, and a single validation split sanity run.

import re
import random

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

final_train_df = clean_train_df.copy()
final_test_df = pd.read_csv(TRAIN_PATH.parent / 'test.csv') if (TRAIN_PATH.parent / 'test.csv').exists() else None

def deberta_light_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' @user ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_keyword(value):
    if pd.isna(value):
        return ''
    return str(value).replace('%20', ' ').strip().lower()

def build_deberta_text(row):
    keyword = normalize_keyword(row.get('keyword', ''))
    text = deberta_light_clean(row['text'])
    if keyword:
        return f'{keyword} [SEP] {text}'
    return text

final_train_df['deberta_text'] = final_train_df.apply(build_deberta_text, axis=1)
if final_test_df is not None:
    final_test_df['deberta_text'] = final_test_df.apply(build_deberta_text, axis=1)

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=192):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def find_best_threshold(y_true, probas):
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in np.arange(0.3, 0.701, 0.01):
        preds = (probas >= threshold).astype(int)
        score = f1_score(y_true, preds)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(np.round(threshold, 2))
    return best_threshold, best_f1

def move_batch_to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}

def evaluate_loader(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            labels = batch['labels'].numpy() if 'labels' in batch else None
            batch = move_batch_to_device(batch)
            outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
            if labels is not None:
                all_labels.extend(labels)
    return np.array(all_probs), np.array(all_labels)

MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN = 192
BATCH_SIZE = 4
EPOCHS = 2
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    final_train_df['deberta_text'],
    final_train_df['target'],
    test_size=0.2,
    random_state=SEED,
    stratify=final_train_df['target'],
)

train_dataset = TweetDataset(train_texts.values, train_labels.values, tokenizer=tokenizer, max_len=MAX_LEN)
valid_dataset = TweetDataset(valid_texts.values, valid_labels.values, tokenizer=tokenizer, max_len=MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loader = None
if final_test_df is not None:
    test_dataset = TweetDataset(final_test_df['deberta_text'].values, tokenizer=tokenizer, max_len=MAX_LEN)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

print('Starting DeBERTa v3 base sanity run...')
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    progress = tqdm(train_loader, desc=f'DeBERTa Epoch {epoch + 1}/{EPOCHS}')
    for batch in progress:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad()
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['labels'])
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / max(len(train_loader), 1)
    valid_probs, valid_true = evaluate_loader(model, valid_loader)
    threshold, val_f1 = find_best_threshold(valid_true, valid_probs)
    print(f'Epoch {epoch + 1}: train_loss={avg_loss:.4f}, valid_f1={val_f1:.4f}, threshold={threshold:.3f}')

valid_probs, valid_true = evaluate_loader(model, valid_loader)
best_threshold, best_f1 = find_best_threshold(valid_true, valid_probs)
valid_preds = (valid_probs >= best_threshold).astype(int)

print('\nDeBERTa v3 base validation results')
print(f'Best threshold: {best_threshold:.3f}')
print(f'Validation F1: {best_f1:.4f}')
print(classification_report(valid_true, valid_preds))
print('Confusion matrix:')
print(confusion_matrix(valid_true, valid_preds))

if test_loader is not None:
    test_probs = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            batch = move_batch_to_device(batch)
            outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1].detach().cpu().numpy()
            test_probs.extend(probs)
    test_preds = (np.array(test_probs) >= best_threshold).astype(int)
    submission_df = pd.DataFrame({'id': final_test_df['id'], 'target': test_preds})
    submission_df.to_csv('deberta_v3_base_submission.csv', index=False)
    print('Saved submission file: deberta_v3_base_submission.csv')
